# Logistic Regression and Regularization

Today (about an hour and a half):

1. **Logistic regression**: predict a *category* (yes/no) instead of a number (25 min demo + 15 min exercise)
2. **Metrics**: is a classifier any good? Accuracy, precision, recall (10 min)
3. **Regularization**: keep a flexible model from overfitting (15 min demo + 15 min exercise)

# Part 1: Logistic regression

Linear regression predicts a **number**. Often we want to predict a **category**: will the passenger survive? Is this email spam? Is this car efficient? With two categories we code the target as $y \in \{0, 1\}$. This is **classification**.

## 1.1 Why not a straight line?

We take the `mpg` cars from Session 1 and define $y = 1$ if the car is *efficient* (highway consumption of at least 25 mpg) and $y = 0$ otherwise. Let's try to predict it from the engine size with a plain linear regression:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression

RAW = "https://raw.githubusercontent.com/cienciadadesdsuab/DS26-27/main/"

mpg = pd.read_csv(RAW + "Laboratoris/01_Introduction%20to%20Python/mpg.csv", index_col=0)
mpg["efficient"] = (mpg["hwy"] >= 25).astype(int)

x = mpg[["displ"]]
y = mpg["efficient"]

line = LinearRegression().fit(x, y)
xs = pd.DataFrame({"displ": np.linspace(1, 8, 100)})

plt.scatter(x, y, edgecolor="k", alpha=0.4)
plt.plot(xs, line.predict(xs), color="red")
plt.xlabel("Engine size (litres)"); plt.ylabel("efficient (0/1)"); plt.ylim(-0.4, 1.4)
plt.show()

The line is a poor fit for this kind of target: it goes **below 0 and above 1**, so it cannot be read as a probability, and it is not what the data look like (a sudden change from 1 to 0, not a slope).

## 1.2 The sigmoid

The fix: keep the linear combination $z = w_0 + w_1 x$ but **squash** it into $(0, 1)$ with the **logistic (sigmoid)** function:

$$\hat p = \sigma(z) = \frac{1}{1 + e^{-z}} \qquad\qquad \hat p = P(y = 1 \mid x)$$

In [ ]:
z = np.linspace(-8, 8, 200)
plt.plot(z, 1 / (1 + np.exp(-z)))
plt.axhline(0.5, color="gray", ls="--"); plt.axvline(0, color="gray", ls="--")
plt.xlabel("z"); plt.ylabel("sigmoid(z)"); plt.show()

* $z$ very positive: $\hat p \to 1$. $z$ very negative: $\hat p \to 0$. $z = 0$: $\hat p = 0.5$.
* The output is a **probability**. To obtain a class we choose a **threshold**, normally 0.5: predict 1 if $\hat p \geq 0.5$, i.e. if $z \geq 0$.
* So the **decision boundary** is where $z = w_0 + w_1 x = 0$.

## 1.3 The loss: log loss

We cannot use the squared error here. For one example with true label $y$ and predicted probability $\hat p$:

$$\text{loss} = -\big[\,y \ln \hat p + (1 - y)\ln(1 - \hat p)\,\big]$$

If $y=1$ the loss is $-\ln \hat p$: small when $\hat p$ is close to 1, huge when the model is confident and wrong.

In [ ]:
for p in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"true label 1, predicted probability {p:4}: loss = {-np.log(p):5.2f}")

Training means finding the $w$ that minimise the average log loss. We do it exactly like before, with **gradient descent**. (The derivation is in the annex; the result is surprisingly simple: the gradient is $(\hat p - y)\,x_j$, the same shape as in linear regression.)

## 1.4 Fitting a logistic regression

With `scikit-learn`. (`C=np.inf` means "no regularization": we will see what `C` is in Part 3.)

In [ ]:
clf = LogisticRegression(C=np.inf).fit(x, y)

w0, w1 = clf.intercept_[0], clf.coef_[0][0]
print(f"z = {w0:.2f} + ({w1:.2f}) * displ")
print(f"decision boundary: displ = {-w0 / w1:.2f} litres")

**Reading the coefficient:** $w_1 < 0$: bigger engines make an efficient car *less* likely. The boundary is at about 3.3 litres.

Let's draw the fitted probability and use it for a prediction:

In [ ]:
plt.scatter(x, y, edgecolor="k", alpha=0.4)
plt.plot(xs, clf.predict_proba(xs)[:, 1], color="red", label="P(efficient)")
plt.axhline(0.5, color="gray", ls="--"); plt.axvline(-w0 / w1, color="gray", ls="--")
plt.xlabel("Engine size (litres)"); plt.ylabel("P(efficient)"); plt.legend(); plt.show()

car = pd.DataFrame({"displ": [2.5]})
print("P(efficient) for a 2.5 L engine:", clf.predict_proba(car)[0, 1].round(3))
print("predicted class:", clf.predict(car)[0])

## 1.5 Under the hood: gradient descent

It is the same function as in the problems, with a new loss. Here is a minimal version (on the standardised input, as we learnt) and a check that it finds the same $w$:

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gradient_descent_logistic(X, y, alpha=0.5, iterations=2000):
    m, n = X.shape
    w = np.zeros(n)
    for _ in range(iterations):
        p = sigmoid(X @ w)
        w = w - alpha * X.T @ (p - y) / m          # gradient of the average log loss
    return w

mu, sd = x["displ"].mean(), x["displ"].std(ddof=0)
X = np.column_stack([np.ones(len(x)), (x["displ"] - mu) / sd])
w_gd = gradient_descent_logistic(X, y.to_numpy())

# convert back to the original units: z = w0 + w1 * (x - mu)/sd
print("gradient descent  :", round(w_gd[0] - w_gd[1] * mu / sd, 3), round(w_gd[1] / sd, 3))
print("sklearn           :", round(w0, 3), round(w1, 3))

## 1.6 Two features: the decision boundary is a line

With two inputs, $z = w_0 + w_1 x_1 + w_2 x_2$ and the boundary $z = 0$ is a **straight line** in the $(x_1, x_2)$ plane. Let's see it with synthetic data: the hours a student studied and slept the night before the exam, and whether they passed.

In [ ]:
rng = np.random.default_rng(3)
n = 120
study = rng.uniform(0, 10, n)
sleep = rng.uniform(3, 10, n)
logit = -9 + 0.9 * study + 0.7 * sleep
passed = (rng.uniform(size=n) < 1 / (1 + np.exp(-logit))).astype(int)

exam = pd.DataFrame({"study": study, "sleep": sleep, "passed": passed})
clf2 = LogisticRegression(C=np.inf).fit(exam[["study", "sleep"]], exam["passed"])

plt.scatter(study[passed == 1], sleep[passed == 1], c="tab:blue", marker="o", edgecolor="k", label="passed")
plt.scatter(study[passed == 0], sleep[passed == 0], c="tab:red", marker="x", label="failed")
b0, (b1, b2) = clf2.intercept_[0], clf2.coef_[0]
xx = np.linspace(0, 10, 50)
plt.plot(xx, -(b0 + b1 * xx) / b2, "k--", label="decision boundary")
plt.ylim(3, 10); plt.xlabel("hours studied"); plt.ylabel("hours slept"); plt.legend(); plt.show()
print("coefficients:", np.round([b0, b1, b2], 2))

Everything on one side of the dashed line is predicted *passed*, the rest *failed*. Both coefficients are positive: studying and sleeping both raise the probability. Points on the wrong side are the model's mistakes.

## ✏️ Exercise 1 (15 min): did the passenger survive?

The Titanic data from Problems 2. We predict `Survived` (1 = survived) from the sex, the class and the age of the passenger. We prepare the data for you: `male` is 1 for men, and the 177 missing ages are filled with the median. We keep 30% of the passengers aside as a validation set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

titanic = pd.read_csv(RAW + "Problemes/Session%202%20-%20Problems/TitanicData.csv")
titanic["male"] = (titanic["Sex"] == "male").astype(int)
titanic["Age"] = titanic["Age"].fillna(titanic["Age"].median())

features = ["male", "Pclass", "Age"]
X_train, X_val, y_train, y_val = train_test_split(
    titanic[features], titanic["Survived"], test_size=0.3, random_state=0, stratify=titanic["Survived"])

print(len(X_train), "training passengers,", len(X_val), "validation passengers")
print("fraction that survived:", round(titanic["Survived"].mean(), 3))

1. Fit a logistic regression on the training set. Use `make_pipeline(StandardScaler(), LogisticRegression(C=np.inf))` (standardising makes the coefficients comparable).
2. Print the coefficients. What does the sign of each one tell you?
3. **Predict** the probability of survival of a 30-year-old woman travelling in first class. And of a 30-year-old man in third class.
4. Compute the **accuracy** on the validation set. Compare it with the accuracy of the lazy model that always predicts "did not survive". Is your model worth it?

In [ ]:
# Your code here

---
*Your answers to 2 and 4 here*

---

## 1.7 Is the classifier any good? Metrics

Accuracy ("the fraction we got right") can mislead. If 95% of the emails are not spam, a model that never flags spam has 95% accuracy and is useless. We need to look at the two kinds of mistakes, in the **confusion matrix**:

| | predicted 0 | predicted 1 |
|---|---|---|
| **actually 0** | true negatives (TN) | false positives (FP) |
| **actually 1** | false negatives (FN) | true positives (TP) |

$$\text{accuracy} = \frac{TP + TN}{\text{all}} \qquad \text{precision} = \frac{TP}{TP + FP} \qquad \text{recall} = \frac{TP}{TP + FN}$$

* **Precision**: of the ones I called positive, how many really were? (Avoid false alarms.)
* **Recall**: of all the real positives, how many did I find? (Avoid missing cases.)

Which one matters more depends on the problem: for a cancer screening we want a high recall, for a spam filter that deletes mail we want a high precision. Let's compute them for the Titanic model:

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score

pred = model.predict(X_val)
tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
print("TN, FP, FN, TP =", tn, fp, fn, tp)
print("accuracy :", round(accuracy_score(y_val, pred), 3), "=", round((tp + tn) / (tn + fp + fn + tp), 3))
print("precision:", round(precision_score(y_val, pred), 3), "=", round(tp / (tp + fp), 3))
print("recall   :", round(recall_score(y_val, pred), 3), "=", round(tp / (tp + fn), 3))

The threshold 0.5 is a choice. Lowering it flags more passengers as survivors: **recall goes up, precision goes down**.

In [ ]:
proba = model.predict_proba(X_val)[:, 1]
for t in [0.3, 0.5, 0.7]:
    pred_t = (proba >= t).astype(int)
    print(f"threshold {t}: precision = {precision_score(y_val, pred_t):.2f} | recall = {recall_score(y_val, pred_t):.2f}")

### ✏️ Mini exercise (3 min)

A model for detecting a disease gets TP = 40, FP = 10, FN = 20, TN = 930. Compute the accuracy, the precision and the recall by hand. Why is the accuracy misleading here?

In [ ]:
# Your code here

---
*Your answer here*

---

# Part 2: Regularization

Last session we saw that a flexible model (a high-degree polynomial) **overfits**: it memorises the noise. We chose a lower degree to avoid it. **Regularization** gives another way: keep the flexible model but **penalise large weights**, because an overfitted curve needs huge, cancelling coefficients to wiggle through every point.

## 2.1 Overfitting, again

The braking distance data from last session, 25 training points and a degree-10 polynomial:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

rng = np.random.default_rng(1)
n = 45
speed = rng.uniform(10, 120, n)
distance = 0.0075 * speed**2 + 0.15 * speed + rng.normal(0, 6, n)

perm = np.random.default_rng(0).permutation(n)
train, val = perm[:25], perm[25:]
x_tr, y_tr = speed[train].reshape(-1, 1), distance[train]
x_va, y_va = speed[val].reshape(-1, 1), distance[val]

def rmse(a, b):
    return np.sqrt(np.mean((a - b) ** 2))

def poly_ridge(degree, alpha):
    return make_pipeline(StandardScaler(), PolynomialFeatures(degree, include_bias=False),
                         StandardScaler(), Ridge(alpha=alpha))

xs = np.linspace(10, 120, 300).reshape(-1, 1)
over = poly_ridge(10, alpha=1e-8).fit(x_tr, y_tr)
print(f"degree 10, no regularization: train RMSE = {rmse(y_tr, over.predict(x_tr)):.1f} | validation RMSE = {rmse(y_va, over.predict(x_va)):.1f}")

## 2.2 The idea: ridge regression

We add a penalty to the cost. For linear regression (**ridge**, or L2):

$$J_{\text{ridge}}(w) = \underbrace{\frac{1}{2m}\sum_i (\hat y_i - y_i)^2}_{\text{error on the data}} \; + \; \underbrace{\lambda \sum_{j \geq 1} w_j^2}_{\text{penalty on big weights}}$$

* $\lambda$ (`alpha` in `scikit-learn`) controls the trade-off: $\lambda = 0$ is the normal fit, a huge $\lambda$ forces all weights to 0 (a flat model).
* The bias $w_0$ is **not** penalised.
* **Standardise the features first**, otherwise the penalty treats the columns unfairly (the penalty depends on the scale of each variable).

The same degree-10 polynomial, with a growing $\lambda$:

In [ ]:
alphas = [1e-8, 1e-2, 1, 100]
fig, axes = plt.subplots(1, len(alphas), sharey=True, figsize=(18, 4))
for a, ax in zip(alphas, axes):
    m = poly_ridge(10, a).fit(x_tr, y_tr)
    ax.scatter(x_tr, y_tr, edgecolor="k"); ax.scatter(x_va, y_va, edgecolor="k", color="orange")
    ax.plot(xs, m.predict(xs), color="red"); ax.set_ylim(-20, 160)
    ax.set_title(f"lambda = {a:g}\ntrain {rmse(y_tr, m.predict(x_tr)):.1f} | val {rmse(y_va, m.predict(x_va)):.1f}")
    ax.set_xlabel("Speed (km/h)")
plt.show()

* **Tiny $\lambda$**: overfitting (wiggly curve, bad validation error).
* **Moderate $\lambda$**: smooth curve, low validation error. The penalty *tamed* the degree-10 polynomial.
* **Huge $\lambda$**: underfitting (a nearly flat line). The weights have been squeezed too much.

We can also see the weights shrinking:

In [ ]:
alphas_sweep = np.logspace(-3, 4, 30)
coefs = np.array([poly_ridge(10, a).fit(x_tr, y_tr)[-1].coef_ for a in alphas_sweep])

plt.semilogx(alphas_sweep, coefs)
plt.xlabel("lambda (alpha)"); plt.ylabel("weight value"); plt.title("Ridge: the weights shrink as lambda grows")
plt.show()

## 2.3 Choosing $\lambda$

$\lambda$ is a **hyperparameter**: we do not learn it from the training data (that would always pick $\lambda=0$). We choose it with the **validation error**, exactly like the degree:

In [ ]:
tr_err = [rmse(y_tr, poly_ridge(10, a).fit(x_tr, y_tr).predict(x_tr)) for a in alphas_sweep]
va_err = [rmse(y_va, poly_ridge(10, a).fit(x_tr, y_tr).predict(x_va)) for a in alphas_sweep]

plt.semilogx(alphas_sweep, tr_err, label="Training")
plt.semilogx(alphas_sweep, va_err, label="Validation")
plt.xlabel("lambda (alpha)"); plt.ylabel("RMSE (m)"); plt.legend(); plt.show()
print("best lambda on validation:", alphas_sweep[int(np.argmin(va_err))].round(4))

The classic U shape for the validation error: too little regularization overfits, too much underfits. With only 20 validation points the estimate is noisy; **cross-validation** (splitting the training data in $k$ folds and averaging) is more robust and is one line in `scikit-learn`:

In [ ]:
from sklearn.model_selection import cross_val_score

for a in [1e-8, 1e-2, 1, 100]:
    scores = cross_val_score(poly_ridge(10, a), x_tr, y_tr, cv=5, scoring="neg_root_mean_squared_error")
    print(f"lambda = {a:g}: 5-fold CV RMSE = {-scores.mean():.2f}")

## 2.4 Regularized logistic regression

The same penalty works in logistic regression: log loss $+\ \lambda \sum w_j^2$. In `scikit-learn` the strength is given by **`C`, which is the *inverse* of $\lambda$**: **small `C` = strong regularization**, large `C` = weak. (That is the `C=np.inf` we used so far: no penalty.)

**A picture first.** Synthetic data with two interleaved half-moons: no straight line can separate them, so we give the model a degree-10 polynomial of the two inputs (65 features) and only 40 training points. The shaded regions show what the model predicts at each point of the plane; the black line is the decision boundary.

In [ ]:
def make_moons(n, noise, rng):
    n1 = n // 2
    t1, t2 = rng.uniform(0, np.pi, n1), rng.uniform(0, np.pi, n - n1)
    X = np.vstack([np.column_stack([np.cos(t1), np.sin(t1)]),
                   np.column_stack([1 - np.cos(t2), 0.5 - np.sin(t2)])]) + rng.normal(0, noise, (n, 2))
    y = np.r_[np.zeros(n1), np.ones(n - n1)]
    p = rng.permutation(n)
    return X[p], y[p]

rng = np.random.default_rng(3)
Xm_train, ym_train = make_moons(40, 0.35, rng)       # few training points
Xm_val, ym_val = make_moons(1000, 0.35, rng)         # large validation set

def moons_model(C):
    return make_pipeline(PolynomialFeatures(10, include_bias=False), StandardScaler(),
                         LogisticRegression(C=C, max_iter=50000))

gx, gy = np.meshgrid(np.linspace(-1.8, 2.8, 250), np.linspace(-1.5, 2.0, 250))
grid = np.column_stack([gx.ravel(), gy.ravel()])

Cs_moons = [1e5, 1, 0.1, 0.001]
fig, axes = plt.subplots(1, len(Cs_moons), sharey=True, figsize=(20, 4))
for C, ax in zip(Cs_moons, axes):
    m = moons_model(C).fit(Xm_train, ym_train)
    prob = m.predict_proba(grid)[:, 1].reshape(gx.shape)
    ax.contourf(gx, gy, prob, levels=[0, 0.5, 1], colors=["#cfe0ff", "#ffd6d6"])
    ax.contour(gx, gy, prob, levels=[0.5], colors="k")
    ax.scatter(Xm_train[ym_train == 0, 0], Xm_train[ym_train == 0, 1], marker="o", c="b", edgecolor="k")
    ax.scatter(Xm_train[ym_train == 1, 0], Xm_train[ym_train == 1, 1], marker="x", c="r")
    ax.set_title(f"C = {C:g}\ntrain {m.score(Xm_train, ym_train):.2f} | val {m.score(Xm_val, ym_val):.2f}")
plt.show()

* **C = 1e5 (no regularization)**: the boundary twists around individual points to get *every* training point right (train accuracy 1.0), but it is a poor boundary for new data: overfitting.
* **C = 1 or 0.1**: a smooth boundary that follows the two moons and generalises better.
* **C = 0.001**: so much penalty that the boundary is almost a plain, gently curved line that misses many points (train 0.82, validation 0.75): underfitting.

Now the same idea on the Titanic with many features: sex, class, age, fare, relatives on board, and all their products and powers up to degree 3 (83 columns!). We use only 150 passengers to train, so the model can overfit.

In [ ]:
more = ["male", "Pclass", "Age", "Fare", "SibSp", "Parch"]
Xa_train, Xa_val, ya_train, ya_val = train_test_split(
    titanic[more], titanic["Survived"], test_size=0.4, random_state=1, stratify=titanic["Survived"])
Xa_train, ya_train = Xa_train.iloc[:150], ya_train.iloc[:150]

def poly_logistic(C):
    return make_pipeline(StandardScaler(), PolynomialFeatures(3, include_bias=False),
                         StandardScaler(), LogisticRegression(C=C, max_iter=20000))

Cs = np.logspace(-3, 5, 9)
tr_acc, va_acc = [], []
for C in Cs:
    m = poly_logistic(C).fit(Xa_train, ya_train)
    tr_acc.append(m.score(Xa_train, ya_train)); va_acc.append(m.score(Xa_val, ya_val))
    print(f"C = {C:9.3g}: train accuracy = {tr_acc[-1]:.3f} | validation accuracy = {va_acc[-1]:.3f}")

plt.semilogx(Cs, tr_acc, "o-", label="Training"); plt.semilogx(Cs, va_acc, "o-", label="Validation")
plt.xlabel("C  (small = strong regularization)"); plt.ylabel("accuracy"); plt.legend(); plt.show()

* **Large C** (almost no regularization): the training accuracy is the highest, but the validation accuracy is worse: overfitting.
* **Very small C**: the model is so constrained that it predicts almost the same for everybody (underfits).
* The best validation accuracy is in between, at a moderate `C`.

## ✏️ Exercise 2 (15 min): choose the regularization for the Titanic

Using the data and `poly_logistic` above:

1. Fit the model with `C = 0.1` and with `C = 1e5`. Which has the higher **training** accuracy? Which has the higher **validation** accuracy? What does that tell you?
2. Look for the best `C` between 0.01 and 10 on the validation set (try at least 10 values).
3. With your chosen `C`, print the confusion matrix on the validation set, and the precision and the recall.
4. **Questions:** (a) Why can't we choose `C` using the training accuracy? (b) A model has a training accuracy of 99% and a validation accuracy of 70%. Is it over or underfitted? What would you change?

In [ ]:
# Your code here

---
*Your answers to 1 and 4 here*

---

## 2.5 Notes

* **L1 regularization (lasso)** penalises $\sum |w_j|$ instead of $\sum w_j^2$. Its special property: it sets some weights **exactly to 0**, so it also *selects variables*. Ridge only shrinks them. In `scikit-learn`: `Lasso(alpha=...)` for regression (logistic regression also has an L1 option).
* Regularization is also the reason to **standardise** the features: the penalty compares the size of the weights.
* More data is the other cure for overfitting: the same flexible model overfits less when there are more points.

# Summary

| | Linear regression | Logistic regression |
|---|---|---|
| Predicts | a number | a probability, then a class (threshold 0.5) |
| Model | $z = w_0 + w_1x_1 + \dots$ | $\hat p = \sigma(z)$ |
| Loss | squared error | log loss |
| Training | least squares / gradient descent | gradient descent |
| Evaluation | RMSE, $R^2$ | accuracy, precision, recall (on validation data!) |

**Regularization**: add $\lambda\sum w_j^2$ to the cost. Small $\lambda$ (large `C`) overfits, huge $\lambda$ (tiny `C`) underfits; choose it with validation or cross-validation.